# Emotions dataset -> binary LTS pipeline (GitHub + Colab)

This notebook follows the same LTS run structure used in the 20 Newsgroups workflow:
1) prepare a binary task,
2) run `main_cluster` style active learning,
3) evaluate the saved checkpoint.

Binary task in this notebook:
- `1` = target emotion (`love` by default)
- `0` = all other emotions


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import sys

REPO_URL = "https://github.com/ShenyaoZhang/DS-GA-3001-Data-Engineering-Project.git"
BRANCH = "emotions-rec-sentiments"

DRIVE_ROOT = "/content/drive/MyDrive"
REPO_ROOT = os.path.join(DRIVE_ROOT, "DS-GA-3001-Data-Engineering-Project")
EXPERIMENT_ROOT = os.path.join(REPO_ROOT, "emotions_rec")

SRC_DIR = os.path.join(EXPERIMENT_ROOT, "src")
PROMPTS_DIR = os.path.join(EXPERIMENT_ROOT, "prompts")
DATA_PROCESSED_DIR = os.path.join(EXPERIMENT_ROOT, "data", "processed")

if not os.path.exists(REPO_ROOT):
    !git clone -b {BRANCH} {REPO_URL} "{REPO_ROOT}"
else:
    %cd "{REPO_ROOT}"
    !git fetch origin
    !git checkout {BRANCH}
    !git pull

%cd "{EXPERIMENT_ROOT}"
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Experiment root:", EXPERIMENT_ROOT)


/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 18 (delta 12), reused 18 (delta 12), pack-reused 0 (from 0)
Unpacking objects: 100% (18/18), 2.40 KiB | 0 bytes/s, done.
From https://github.com/ShenyaoZhang/DS-GA-3001-Data-Engineering-Project
   8609b21..433e545  emotions-rec-sentiments -> origin/emotions-rec-sentiments
Already on 'emotions-rec-sentiments'
Your branch is behind 'origin/emotions-rec-sentiments' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating 8609b21..433e545
Fast-forward
 emotions_rec/COLAB.md                              | 11 +--
 emotions_rec/README.md                             |  9 +-
 .../notebooks/emotions_rec_sentiment_repro.ipynb   | 97 ++++++++++------------
 emotions_rec/run_configs/sentiment_random_run.txt  |  5 +-
 .../run_configs/sentiment_thompson_run.tx

In [ ]:
from getpass import getpass
import os

token = getpass("Enter your HF token: ")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token
print("HF token set.")


Enter your HF token: ··········
HF token set.


In [ ]:
!pip -q install -r "{os.path.join(REPO_ROOT, 'requirements.txt')}"
!pip -q install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 107.2 MB/s eta 0:00:00


In [ ]:
TARGET_LABEL = "love"

%cd "{EXPERIMENT_ROOT}"
!python -u scripts/prepare_emotions_binary.py --label {TARGET_LABEL}

TRAIN_FILE = f"data/processed/emotions_{TARGET_LABEL}_smoke_train"
VAL_FILE = f"data/processed/emotions_{TARGET_LABEL}_smoke_validation.csv"
TEST_FILE = f"data/processed/emotions_{TARGET_LABEL}_test.csv"

print("Train:", TRAIN_FILE)
print("Val:  ", VAL_FILE)
print("Test: ", TEST_FILE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project/emotions_rec/data/processed/train_inner_emotions_sentiment.csv
label
0    8762
1     572
2    6666
Name: count, dtype: int64


In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_FILE + ".csv")
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

print("train distribution:", train_df["label"].value_counts().to_dict())
print("val distribution:", val_df["label"].value_counts().to_dict())
print("test distribution:", test_df["label"].value_counts().to_dict())


Saved: /content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project/emotions_rec/prompts/few_shot_examples_sentiment.json
Num examples: 9


In [ ]:
EXPECTED = [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "main_cluster_emotion_binary.py",
    "eval_emotion_binary.py",
]

missing = []
for name in EXPECTED:
    path = os.path.join(SRC_DIR, name)
    ok = os.path.exists(path)
    print(f"{name}: {'FOUND' if ok else 'MISSING'}")
    if not ok:
        missing.append(name)

script_ok = os.path.exists(os.path.join(EXPERIMENT_ROOT, "scripts", "prepare_emotions_binary.py"))
print("scripts/prepare_emotions_binary.py:", "FOUND" if script_ok else "MISSING")
if not script_ok:
    missing.append("scripts/prepare_emotions_binary.py")

if missing:
    raise FileNotFoundError(f"Missing: {missing}")


preprocessing.py: FOUND
LDA.py: FOUND
labeling.py: FOUND
random_sampling.py: FOUND
thompson_sampling.py: FOUND
fine_tune.py: FOUND
sentiment_labels.py: FOUND
main_cluster_sentiment.py: FOUND


In [ ]:
import subprocess

targets = [os.path.join(SRC_DIR, f) for f in EXPECTED]
res = subprocess.run(["python", "-m", "py_compile", *targets], capture_output=True, text=True, cwd=EXPERIMENT_ROOT)
if res.returncode == 0:
    print("All files compiled successfully.")
else:
    print(res.stderr)


All files compiled successfully.


In [ ]:
%cd "{EXPERIMENT_ROOT}"

# main_cluster-style command with a short budget (~30 min target)
!python -u src/main_cluster_emotion_binary.py \
  -sample_size 200 \
  -filename "{TRAIN_FILE}" \
  -val_path "{VAL_FILE}" \
  -balance False \
  -sampling thompson \
  -filter_label True \
  -model_finetune bert-base-uncased \
  -labeling qwen \
  -model text \
  -baseline 0.10 \
  -metric f1_pos \
  -cluster_size 8 \
  -positive_label "{TARGET_LABEL}" \
  -hf_model_id "Qwen/Qwen2.5-3B-Instruct" \
  -max_iterations 3 \
  -num_train_epochs 2 \
  -max_length 128 \
  -batch_size 16 \
  -confidence_threshold 0.40


/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project/emotions_rec
Validation sentiment labels: {0: 1037, 2: 963}
using data saved on disk
Loading weights: 100% 199/199 [00:00<00:00, 1228.61it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored w

## Evaluate the trained binary model

After training, pick your best saved checkpoint from `models/`.

This prints:
- positive-class precision/recall/F1
- macro-F1
- accuracy
- mean confidence

In [14]:
%cd "/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project/emotions_rec"
!ls -la models

/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project/emotions_rec
total 148
drwx------ 2 root root 4096 May  5 01:57 binary_fine_tunned_0_bandit_1
drwx------ 2 root root 4096 May  5 00:22 binary_fine_tunned_0_bandit_3
drwx------ 2 root root 4096 May  5 01:16 binary_fine_tunned_0_bandit_4
drwx------ 2 root root 4096 May  5 01:17 binary_fine_tunned_1_bandit_4
drwx------ 2 root root 4096 May  5 01:21 binary_fine_tunned_3_bandit_7
drwx------ 2 root root 4096 May  4 23:28 fine_tunned_0_bandit_1
drwx------ 2 root root 4096 May  4 21:15 fine_tunned_0_bandit_3
drwx------ 2 root root 4096 May  4 21:46 fine_tunned_0_bandit_5
drwx------ 2 root root 4096 May  4 21:05 fine_tunned_0_bandit_6
drwx------ 2 root root 4096 May  4 19:47 fine_tunned_1_bandit_5
drwx------ 2 root root 4096 May  4 20:37 fine_tunned_1_bandit_9
drwx------ 2 root root 4096 May  4 21:24 fine_tunned_2_bandit_0
drwx------ 2 root root 4096 May  4 21:49 fine_tunned_2_bandit_5
drwx------ 2 root root 4096 May  4 23:31 fine_tunne

In [15]:
%cd "/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project/emotions_rec"
!python -u src/eval_emotion_binary.py \
  -test_path "{TEST_FILE}" \
  -model_path "models/binary_love_fine_tunned_0_bandit_0" \
  -target_emotion "{TARGET_LABEL}" \
  -base_model "bert-base-uncased" \
  -max_length 128

/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project/emotions_rec
Loading weights: 100% 201/201 [00:00<00:00, 553.27it/s, Materializing param=classifier.weight]

=== Sentiment evaluation ===
              precision    recall  f1-score   support

    negative       0.82      0.80      0.81      1080
     neutral       0.08      0.09      0.09        66
    positive       0.75      0.77      0.76       854

    accuracy                           0.77      2000
   macro avg       0.55      0.55      0.55      2000
weighted avg       0.77      0.77      0.77      2000

Accuracy:    0.7650
F1 Macro:    0.5534
F1 Weighted: 0.7663
Mean confidence: 0.9626
